# Generate GPT-2 Joke Arena v3

This performs **real GPT-2 inference** and generates **160 stories**:

## Standard — 80 stories
For each of 10 prompts:
- 2 GPT-2 Small
- 2 GPT-2 Medium
- 2 GPT-2 Large
- 2 GPT-2 XL

Standard stories:
- seed **1337**
- temperature `0.95`
- top-p `0.95`
- top-k `50`
- **300-word cap**
- natural EOS is allowed, so they can stop early

## Telephone — 80 stories
For each of the same 10 prompts:
- 2 GPT-2 Small
- 2 GPT-2 Medium
- 2 GPT-2 Large
- 2 GPT-2 XL

Each telephone story is **300 words = 4 × 75-word hops**.

- hop 1 sees the original prompt
- hop 2 sees only hop 1
- hop 3 sees only hop 2
- hop 4 sees only hop 3
- the displayed story concatenates all four hops so you can see the drift

EOS is suppressed *inside telephone hops only* so every telephone message reaches 75 words. Standard stories still use natural EOS.

For each model, the standard stream is seeded once to 1337 and generated sequentially. Then the telephone stream is separately reseeded to 1337 and generated sequentially.


In [ ]:
!pip -q install -U "transformers>=4.45" accelerate safetensors

In [ ]:
import gc, json, random, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
PROMPTS = ['I went to the store yesterday, but', 'When I opened the refrigerator this morning,', 'The new employee seemed completely normal until', 'Nobody at the restaurant could explain why', 'My neighbor knocked on my door and asked if', 'The package arrived three weeks late, and', 'I knew the hotel was unusual when', 'At first the job interview was going well, but', 'The sign on the door clearly said', 'I called customer service because']

In [ ]:
MODELS = [('small', 'GPT-2 Small', 'openai-community/gpt2'), ('medium', 'GPT-2 Medium', 'openai-community/gpt2-medium'), ('large', 'GPT-2 Large', 'openai-community/gpt2-large'), ('xl', 'GPT-2 XL', 'openai-community/gpt2-xl')]

In [ ]:
SEED = 1337
TEMPERATURE = 0.95
TOP_P = 0.95
TOP_K = 50

STANDARD_WORD_CAP = 300
STANDARD_MAX_NEW_TOKENS = 700

TELEPHONE_WORD_CAP = 300
TELEPHONE_HOPS = 4
TELEPHONE_HOP_WORDS = 75

OUTPUT = Path("joke_bank.json")
CHECKPOINT = Path("joke_bank_checkpoint_v3.json")


## Generation helpers

In [ ]:

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def word_count(text):
    return len(re.findall(r"\S+", text))

def cut_words(text, n):
    matches = list(re.finditer(r"\S+", text))
    if len(matches) <= n:
        return text
    return text[:matches[n-1].end()]

def load_model(model_id):
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    tok.truncation_side = "left"

    mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
    )
    mdl.eval()
    return tok, mdl

@torch.inference_mode()
def standard_story(tokenizer, model, prompt):
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    device = next(model.parameters()).device
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    t0 = time.perf_counter()
    out = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=STANDARD_MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    dt = time.perf_counter() - t0

    new_ids = out[0, input_ids.shape[1]:]
    continuation = tokenizer.decode(
        new_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    text = prompt + continuation

    natural_eos = (
        tokenizer.eos_token_id is not None
        and tokenizer.eos_token_id in new_ids.tolist()
    )

    hit_cap = word_count(text) > STANDARD_WORD_CAP
    if hit_cap:
        text = cut_words(text, STANDARD_WORD_CAP)

    return {
        "text": text,
        "generation_type": "standard",
        "word_cap": STANDARD_WORD_CAP,
        "actual_words": word_count(text),
        "generated_tokens": int(len(new_ids)),
        "natural_eos": bool(natural_eos),
        "stop_reason": "eos" if natural_eos else ("word_cap" if hit_cap else "token_cap"),
        "generation_seconds": dt,
    }

@torch.inference_mode()
def telephone_hop(tokenizer, model, context, target_words=75):
    """
    Produce exactly target_words of visible text where possible.
    EOS is suppressed because telephone stories are intentionally
    fixed 4x75-word relay chains.
    """
    built = ""
    total_tokens = 0

    while word_count(built) < target_words:
        # Keep only recent context if necessary.
        enc = tokenizer(
            context + built,
            return_tensors="pt",
            add_special_tokens=False,
            truncation=True,
            max_length=800,
        )
        device = next(model.parameters()).device
        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)

        kwargs = {}
        if tokenizer.eos_token_id is not None:
            kwargs["bad_words_ids"] = [[tokenizer.eos_token_id]]

        out = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=140,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            repetition_penalty=1.0,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
            **kwargs,
        )

        new_ids = out[0, input_ids.shape[1]:]
        total_tokens += int(len(new_ids))

        piece = tokenizer.decode(
            new_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        if not piece:
            break

        built += piece

    return cut_words(built, target_words), total_tokens

def telephone_story(tokenizer, model, prompt):
    """
    300 displayed words = four 75-word relay hops.

    Each hop sees only the previous hop, never the older chain.
    """
    t0 = time.perf_counter()
    context = prompt
    hops = []
    total_tokens = 0

    for hop_number in range(1, TELEPHONE_HOPS + 1):
        hop_text, n_tokens = telephone_hop(
            tokenizer,
            model,
            context,
            TELEPHONE_HOP_WORDS,
        )
        hops.append(hop_text)
        total_tokens += n_tokens

        # The telephone operation:
        context = hop_text

    text = "\n\n".join(h.strip() for h in hops if h.strip())
    text = cut_words(text, TELEPHONE_WORD_CAP)

    return {
        "text": text,
        "generation_type": "telephone",
        "word_cap": TELEPHONE_WORD_CAP,
        "actual_words": word_count(text),
        "generated_tokens": total_tokens,
        "natural_eos": False,
        "stop_reason": "telephone_4x75",
        "telephone_hops": TELEPHONE_HOPS,
        "telephone_hop_words": TELEPHONE_HOP_WORDS,
        "generation_seconds": time.perf_counter() - t0,
    }


## Generate all 160 stories

In [ ]:

if CHECKPOINT.exists():
    generated = json.loads(CHECKPOINT.read_text(encoding="utf-8"))
else:
    generated = {}

for difficulty, model_label, model_id in MODELS:
    print("\n" + "="*90)
    print(model_label, "-", model_id)
    print("="*90)

    tokenizer, model = load_model(model_id)

    # STANDARD stream: seed once at 1337, then generate sequentially.
    missing_standard = any(
        f"p{pi:02d}_{difficulty}_standard_{rep}" not in generated
        for pi in range(1, 11) for rep in [1, 2]
    )
    if missing_standard:
        seed_everything(SEED)

        for pi, prompt in enumerate(PROMPTS, start=1):
            pid = f"p{pi:02d}"
            for rep in [1, 2]:
                aid = f"{pid}_{difficulty}_standard_{rep}"

                if aid in generated:
                    # Consume the RNG in the same way to preserve later deterministic outputs.
                    _ = standard_story(tokenizer, model, prompt)
                    print(aid, "cached")
                    continue

                result = standard_story(tokenizer, model, prompt)
                generated[aid] = {
                    "answer_id": aid,
                    "prompt_id": pid,
                    "difficulty": difficulty,
                    "model_label": model_label,
                    "model_id": model_id,
                    "replicate": rep,
                    "seed": SEED,
                    **result,
                }
                CHECKPOINT.write_text(
                    json.dumps(generated, ensure_ascii=False, indent=2),
                    encoding="utf-8",
                )
                print(
                    f"{aid}: {result['actual_words']}w, "
                    f"{result['stop_reason']}, {result['generation_seconds']:.1f}s"
                )
    else:
        print("standard stream already complete")

    # TELEPHONE stream: separately reseed to 1337, then generate sequentially.
    missing_telephone = any(
        f"p{pi:02d}_{difficulty}_telephone_{rep}" not in generated
        for pi in range(1, 11) for rep in [1, 2]
    )
    if missing_telephone:
        seed_everything(SEED)

        for pi, prompt in enumerate(PROMPTS, start=1):
            pid = f"p{pi:02d}"
            for rep in [1, 2]:
                aid = f"{pid}_{difficulty}_telephone_{rep}"

                if aid in generated:
                    _ = telephone_story(tokenizer, model, prompt)
                    print(aid, "cached")
                    continue

                result = telephone_story(tokenizer, model, prompt)
                generated[aid] = {
                    "answer_id": aid,
                    "prompt_id": pid,
                    "difficulty": difficulty,
                    "model_label": model_label,
                    "model_id": model_id,
                    "replicate": rep,
                    "seed": SEED,
                    **result,
                }
                CHECKPOINT.write_text(
                    json.dumps(generated, ensure_ascii=False, indent=2),
                    encoding="utf-8",
                )
                print(
                    f"{aid}: {result['actual_words']}w, "
                    f"{result['generation_seconds']:.1f}s"
                )
    else:
        print("telephone stream already complete")

    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nTOTAL:", len(generated))
assert len(generated) == 160


## Build and validate `joke_bank.json`

In [ ]:

bank = {
    "schema_version": 3,
    "generated": True,
    "generation_settings": {
        "seed": SEED,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "top_k": TOP_K,
        "standard_word_cap": STANDARD_WORD_CAP,
        "standard_eos": "natural",
        "telephone_word_cap": TELEPHONE_WORD_CAP,
        "telephone_hops": TELEPHONE_HOPS,
        "telephone_hop_words": TELEPHONE_HOP_WORDS,
        "telephone_context": "hop N receives only hop N-1",
    },
    "models": [
        {"difficulty": d, "label": label, "model_id": mid}
        for d, label, mid in MODELS
    ],
    "prompts": [],
}

for pi, prompt in enumerate(PROMPTS, start=1):
    pid = f"p{pi:02d}"
    prompt_answers = [
        a for a in generated.values()
        if a["prompt_id"] == pid
    ]

    bank["prompts"].append({
        "prompt_id": pid,
        "text": prompt,
        "answers": prompt_answers,
    })

all_answers = [a for p in bank["prompts"] for a in p["answers"]]
standard = [a for a in all_answers if a["generation_type"] == "standard"]
telephone = [a for a in all_answers if a["generation_type"] == "telephone"]

assert len(all_answers) == 160
assert len(standard) == 80
assert len(telephone) == 80

for p in bank["prompts"]:
    std = [a for a in p["answers"] if a["generation_type"] == "standard"]
    tel = [a for a in p["answers"] if a["generation_type"] == "telephone"]

    assert len(std) == 8
    assert len(tel) == 8

    for group in [std, tel]:
        counts = {}
        for a in group:
            counts[a["difficulty"]] = counts.get(a["difficulty"], 0) + 1
        assert counts == {"small": 2, "medium": 2, "large": 2, "xl": 2}

OUTPUT.write_text(
    json.dumps(bank, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Wrote", OUTPUT, "-", round(OUTPUT.stat().st_size/1024, 1), "KB")


## Summary

In [ ]:
df = pd.DataFrame(list(generated.values()))

display(
    df.groupby(["generation_type", "model_label"]).agg(
        answers=("answer_id", "count"),
        median_words=("actual_words", "median"),
        min_words=("actual_words", "min"),
        max_words=("actual_words", "max"),
        mean_seconds=("generation_seconds", "mean"),
        natural_eos_rate=("natural_eos", "mean"),
    ).reset_index()
)

print("Standard stories:", (df["generation_type"] == "standard").sum())
print("Telephone stories:", (df["generation_type"] == "telephone").sum())
print("Total generation minutes:", round(df["generation_seconds"].sum()/60, 2))


## Inspect a few telephone stories

In [ ]:
shown = 0
for a in generated.values():
    if a["generation_type"] == "telephone":
        print("\n" + "="*100)
        print(a["answer_id"], "|", a["model_label"], "|", a["actual_words"], "words")
        print("="*100)
        print(a["text"])
        shown += 1
        if shown >= 8:
            break


## Download the finished bank

In [ ]:
from google.colab import files
files.download("joke_bank.json")
